In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Layer, Dense, Dropout, Input, Flatten, LSTM, GRU
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (f1_score, accuracy_score, matthews_corrcoef,
                             confusion_matrix, roc_auc_score, average_precision_score)
from imblearn.over_sampling import RandomOverSampler
import time, warnings, os, random
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    tf.keras.utils.set_random_seed(seed)

set_seed(42)
print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


In [2]:
df = pd.read_csv(r"C:\Users\nuzha\IDD_project\de2c6f75dd50d933_MOHANAD_A4706\de2c6f75dd50d933_MOHANAD_A4706\data\NF-BoT-IoT.csv")  # change path if needed

df = df.fillna(0)
df.replace([np.inf, -np.inf], 0, inplace=True)

# Feature Engineering
df['IN_BYTE_PER_PKT']  = df['IN_BYTES']  / (df['IN_PKTS']  + 1e-6)
df['OUT_BYTE_PER_PKT'] = df['OUT_BYTES'] / (df['OUT_PKTS'] + 1e-6)
df['IN_PKT_RATE']      = df['IN_PKTS']   / (df['FLOW_DURATION_MILLISECONDS'] + 1e-6)
df['OUT_PKT_RATE']     = df['OUT_PKTS']  / (df['FLOW_DURATION_MILLISECONDS'] + 1e-6)
df['TOTAL_PKTS']       = df['IN_PKTS']   + df['OUT_PKTS']
df['TOTAL_BYTES']      = df['IN_BYTES']  + df['OUT_BYTES']
df['FLOW_DUR_PER_PKT'] = df['FLOW_DURATION_MILLISECONDS'] / (df['TOTAL_PKTS'] + 1e-6)

# TCP Flag features
df['TCP_SYN'] = df['TCP_FLAGS'].apply(lambda x: 1 if int(x) & 0x02 else 0)
df['TCP_ACK'] = df['TCP_FLAGS'].apply(lambda x: 1 if int(x) & 0x10 else 0)
df['TCP_FIN'] = df['TCP_FLAGS'].apply(lambda x: 1 if int(x) & 0x01 else 0)

# Binary features for common L7 protocols
df['IS_DNS']   = (df['L7_PROTO'] == 53).astype(int)
df['IS_HTTP']  = (df['L7_PROTO'] == 80).astype(int)
df['IS_HTTPS'] = (df['L7_PROTO'] == 443).astype(int)

# Interaction Features
df['IN_OUT_BYTES_RATIO'] = df['IN_BYTES'] / (df['OUT_BYTES'] + 1e-6)
df['IN_OUT_PKTS_RATIO']  = df['IN_PKTS']  / (df['OUT_PKTS']  + 1e-6)

# Label encoding
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Label'] = le.fit_transform(df['Label'])

# Final feature list
exclude = ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'Label', 'Attack']
feature_cols = [c for c in df.columns if c not in exclude]

print("Number of features:", len(feature_cols))
print(feature_cols)

print("Shape:", df.shape)
print(df['Attack'].value_counts())

Number of features: 25
['L4_SRC_PORT', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'OUT_BYTES', 'IN_PKTS', 'OUT_PKTS', 'TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'IN_BYTE_PER_PKT', 'OUT_BYTE_PER_PKT', 'IN_PKT_RATE', 'OUT_PKT_RATE', 'TOTAL_PKTS', 'TOTAL_BYTES', 'FLOW_DUR_PER_PKT', 'TCP_SYN', 'TCP_ACK', 'TCP_FIN', 'IS_DNS', 'IS_HTTP', 'IS_HTTPS', 'IN_OUT_BYTES_RATIO', 'IN_OUT_PKTS_RATIO']
Shape: (600100, 29)
Attack
Reconnaissance    470655
DDoS               56844
DoS                56833
Benign             13859
Theft               1909
Name: count, dtype: int64


In [3]:
# Check the exact name from the value_counts above
zero_day_attacks = ['DoS']          # adjust if the name is 'dos' or 'DDoS' etc.

df = df.reset_index(drop=True)

zero_day_df = df[df['Attack'].str.lower().isin([a.lower() for a in zero_day_attacks])].copy()
normal_df   = df[~df['Attack'].str.lower().isin([a.lower() for a in zero_day_attacks])].copy()

cutoff1 = int(0.80 * len(normal_df))
dev_df      = normal_df.iloc[:cutoff1].copy()
test_normal = normal_df.iloc[cutoff1:].copy()

cutoff2 = int(0.90 * len(dev_df))
train_df = dev_df.iloc[:cutoff2].copy()
val_df   = dev_df.iloc[cutoff2:].copy()

print("Train     :", train_df.shape)
print("Validation:", val_df.shape)
print("Known Test:", test_normal.shape)
print("Zero-day  :", zero_day_df.shape)
print("Zero-day leakage (must be 0):",
      train_df[train_df['Attack'].str.lower().isin([a.lower() for a in zero_day_attacks])].shape[0],
      val_df[val_df['Attack'].str.lower().isin([a.lower() for a in zero_day_attacks])].shape[0])

Train     : (391151, 29)
Validation: (43462, 29)
Known Test: (108654, 29)
Zero-day  : (56833, 29)
Zero-day leakage (must be 0): 0 0


In [4]:
def create_sequences(data, feature_cols, window_size=10, stride=5):
    sequences, labels, attacks = [], [], []
    feats = data[feature_cols].values.astype(np.float32)
    labs  = data['Label'].values
    atks  = data['Attack'].values
    for i in range(0, len(feats) - window_size + 1, stride):
        sequences.append(feats[i:i+window_size])
        labels.append(labs[i + window_size - 1])
        attacks.append(atks[i + window_size - 1])
    return np.array(sequences, dtype=np.float32), np.array(labels), np.array(attacks)

def safe_create_sequences(data, feature_cols, window_size=10, stride=5):
    if len(data) < window_size:
        return (np.empty((0, window_size, len(feature_cols)), dtype=np.float32),
                np.empty((0,), dtype=np.int64),
                np.empty((0,), dtype=object))
    return create_sequences(data, feature_cols, window_size, stride)

WINDOW, STRIDE = 10, 5

X_train, y_train, _ = create_sequences(train_df, feature_cols, WINDOW, STRIDE)
X_val,   y_val,   _ = create_sequences(val_df,   feature_cols, WINDOW, STRIDE)
X_test_known, y_test_known, atk_known = create_sequences(test_normal, feature_cols, WINDOW, STRIDE)

# Zero-day sequences
X_dos, y_dos, atk_dos = safe_create_sequences(zero_day_df, feature_cols, WINDOW, STRIDE)

print("DoS sequences:", X_dos.shape)

X_test = np.concatenate([X_test_known, X_dos], axis=0)
y_test = np.concatenate([y_test_known, y_dos], axis=0)
attack_test = np.concatenate([atk_known, atk_dos], axis=0)

print("Final Test shape:", X_test.shape)

DoS sequences: (11365, 10, 25)
Final Test shape: (33094, 10, 25)


In [5]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train.reshape(-1, X_train.shape[-1])).reshape(X_train.shape)
X_val   = scaler.transform(X_val.reshape(-1, X_val.shape[-1])).reshape(X_val.shape)
X_test  = scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)

n_s, t, f = X_train.shape
X_train_2d, y_train = RandomOverSampler(random_state=42).fit_resample(X_train.reshape(n_s, -1), y_train)
X_train = X_train_2d.reshape(-1, t, f)

n_features = X_train.shape[-1]
print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)
print("n_features =", n_features)

Train: (154140, 10, 25) | Val: (8691, 10, 25) | Test: (33094, 10, 25)
n_features = 25


In [6]:
# Paste StandardESN, HarmonicESN, FB_HESN_Layer exactly as before

def evaluate_and_store(y_true, y_pred, y_prob, attack_test, model, model_name, results_list):
    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred, average='macro')
    mcc  = matthews_corrcoef(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)
    roc = roc_auc_score(y_true, y_prob)
    pr  = average_precision_score(y_true, y_prob)

    # BoT zero-day
    mask = np.array([str(a).lower() == 'dos' for a in attack_test])
    total = mask.sum()
    detected = (y_pred[mask] == 1).sum() if total > 0 else 0
    dos_rate = detected / total if total > 0 else np.nan

    total_params = model.count_params()
    trainable = int(np.sum([np.prod(v.shape) for v in model.trainable_weights]))

    _ = model.predict(X_test[:128], verbose=0)
    t0 = time.time()
    _ = model.predict(X_test, batch_size=256, verbose=0)
    infer_time = time.time() - t0

    results_list.append({
        'Model': model_name,
        'Accuracy': round(acc,4),
        'Macro-F1': round(f1,4),
        'MCC': round(mcc,4),
        'FPR': round(fpr,4),
        'FNR': round(fnr,4),
        'ROC-AUC': round(roc,4),
        'PR-AUC': round(pr,4),
        'DoS': f"{detected}/{total} ({dos_rate:.4f})",
        'ZD-MacroRecall': round(dos_rate,4),
        'Total Params': total_params,
        'Trainable Params': trainable,
        'Inference Time (s)': round(infer_time,2)
    })
    print(f"{model_name} done | Macro-F1={f1:.4f} | DoS={dos_rate:.4f}")

In [7]:
class StandardESN(Layer):
    def __init__(self, units=64, spectral_radius=0.9, leak_rate=0.3, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.spectral_radius = spectral_radius
        self.leak_rate = leak_rate
    def build(self, input_shape):
        feature_dim = int(input_shape[-1])
        self.W_in = self.add_weight(shape=(feature_dim, self.units), initializer='glorot_uniform', trainable=False)
        W = np.random.uniform(-1,1,(self.units,self.units)).astype(np.float32)
        W *= (np.random.rand(*W.shape) < 0.1).astype(np.float32)
        rho = np.max(np.abs(np.linalg.eigvals(W)))
        W *= (self.spectral_radius / (rho + 1e-8))
        self.W = self.add_weight(shape=(self.units,self.units), initializer=tf.constant_initializer(W), trainable=False)
        self.bias = self.add_weight(shape=(self.units,), initializer='zeros', trainable=False)
        super().build(input_shape)
    def call(self, inputs):
        batch = tf.shape(inputs)[0]
        state = tf.zeros((batch, self.units))
        def body(t, state):
            pre = tf.matmul(inputs[:,t,:], self.W_in) + tf.matmul(state, self.W) + self.bias
            return t+1, (1-self.leak_rate)*state + self.leak_rate*tf.tanh(pre)
        _, final = tf.while_loop(lambda t,s: t < tf.shape(inputs)[1], body, (0, state))
        return final

class HarmonicESN(Layer):
    def __init__(self, units=64, spectral_radius=0.9, leak_rate=0.3, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.spectral_radius = spectral_radius
        self.leak_rate = leak_rate
    def build(self, input_shape):
        feature_dim = int(input_shape[-1])
        self.W_in = self.add_weight(shape=(feature_dim, self.units), initializer='glorot_uniform', trainable=False)
        W = np.random.uniform(-1,1,(self.units,self.units)).astype(np.float32)
        W *= (np.random.rand(*W.shape) < 0.1).astype(np.float32)
        rho = np.max(np.abs(np.linalg.eigvals(W)))
        W *= (self.spectral_radius / (rho + 1e-8))
        self.W = self.add_weight(shape=(self.units,self.units), initializer=tf.constant_initializer(W), trainable=False)
        self.bias = self.add_weight(shape=(self.units,), initializer='zeros', trainable=False)
        super().build(input_shape)
    def call(self, inputs):
        batch = tf.shape(inputs)[0]
        state = tf.zeros((batch, self.units))
        def body(t, state):
            pre = tf.matmul(inputs[:,t,:], self.W_in) + tf.matmul(state, self.W) + self.bias
            return t+1, (1-self.leak_rate)*state + self.leak_rate*tf.sin(pre)
        _, final = tf.while_loop(lambda t,s: t < tf.shape(inputs)[1], body, (0, state))
        return final

class FB_HESN_Layer(Layer):
    def __init__(self, units=64, spectral_radius=0.9, leak_rate=0.3, omega_min=0.3, omega_max=3.5, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.spectral_radius = spectral_radius
        self.leak_rate = leak_rate
        self.omega_min = omega_min
        self.omega_max = omega_max
    def build(self, input_shape):
        feature_dim = int(input_shape[-1])
        self.W_in = self.add_weight(shape=(feature_dim, self.units), initializer='glorot_uniform', trainable=False)
        W = np.random.uniform(-1,1,(self.units,self.units)).astype(np.float32)
        W *= (np.random.rand(*W.shape) < 0.1).astype(np.float32)
        rho = np.max(np.abs(np.linalg.eigvals(W)))
        W *= (self.spectral_radius / (rho + 1e-8))
        self.W = self.add_weight(shape=(self.units,self.units), initializer=tf.constant_initializer(W), trainable=False)
        self.bias = self.add_weight(shape=(self.units,), initializer='zeros', trainable=False)
        omega = np.random.uniform(self.omega_min, self.omega_max, self.units).astype(np.float32)
        self.omega = self.add_weight(shape=(self.units,), initializer=tf.constant_initializer(omega), trainable=False)
        super().build(input_shape)
    def call(self, inputs):
        batch = tf.shape(inputs)[0]
        state = tf.zeros((batch, self.units))
        def body(t, state):
            pre = tf.matmul(inputs[:,t,:], self.W_in) + tf.matmul(state, self.W) + self.bias
            return t+1, (1-self.leak_rate)*state + self.leak_rate*tf.sin(self.omega * pre)
        _, final = tf.while_loop(lambda t,s: t < tf.shape(inputs)[1], body, (0, state))
        return final



In [8]:
results = []

def train_neural(build_fn, name):
    tf.keras.backend.clear_session()
    set_seed(42)
    model = build_fn()
    model.compile(optimizer=Adam(0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=40, batch_size=256,
              callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0)],
              verbose=1)
    y_prob = model.predict(X_test, batch_size=256, verbose=0)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    evaluate_and_store(y_test, y_pred, y_prob, attack_test, model, name, results)

def build_standard_esn():
    inputs = Input(shape=(10, n_features))
    x = StandardESN(units=64)(inputs)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.3)(x)
    return Model(inputs, Dense(2, activation='softmax')(x))

def build_harmonic_esn():
    inputs = Input(shape=(10, n_features))
    x = HarmonicESN(units=64)(inputs)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.3)(x)
    return Model(inputs, Dense(2, activation='softmax')(x))

def build_fb_hesn():
    inputs = Input(shape=(10, n_features))
    x = FB_HESN_Layer(units=64, spectral_radius=0.9, leak_rate=0.3,
                      omega_min=0.3, omega_max=1.0)(inputs)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.3)(x)
    return Model(inputs, Dense(2, activation='softmax')(x))

def build_gru():
    inputs = Input(shape=(10, n_features))
    x = GRU(64)(inputs)
    x = Dropout(0.3)(x)
    x = Dense(32, activation='relu')(x)
    return Model(inputs, Dense(2, activation='softmax')(x))

train_neural(build_standard_esn, "Standard ESN-64")
train_neural(build_harmonic_esn, "Harmonic ESN-64")
train_neural(build_fb_hesn, "FB-HESN-64")
train_neural(build_gru, "GRU-64")

df_master = pd.DataFrame(results)
print(df_master.to_string(index=False))
df_master.to_csv("BoT_master_64.csv", index=False)


Epoch 1/40
603/603 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.9597 - loss: 0.1282 - val_accuracy: 0.8047 - val_loss: 1.2839
Epoch 2/40
603/603 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.9800 - loss: 0.0558 - val_accuracy: 0.8062 - val_loss: 1.1024
Epoch 3/40
603/603 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.9838 - loss: 0.0452 - val_accuracy: 0.8075 - val_loss: 1.1211
Epoch 4/40
603/603 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.9856 - loss: 0.0404 - val_accuracy: 0.8087 - val_loss: 1.1833
Epoch 5/40
603/603 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.9866 - loss: 0.0378 - val_accuracy: 0.8093 - val_loss: 1.1294
Epoch 6/40
603/603 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.9870 - loss: 0.0358 - val_accuracy: 0.8097 - val_loss: 1.1355
Epoch 7/40
603/603 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.9875 - loss: 0.0341 - val_accuracy: 0.8105 - val_loss: 1.1762
Standard ESN-64 done | Macro-F1=0.5993 | DoS=0.9960
Epoch 1/40
603/603 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/ste